In [ ]:
import pandas as pd

# 위키피디아의 S&P 500 구성 종목 목록 URL
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

# HTML 테이블을 읽어오기
tables = pd.read_html(url)
df = tables[0]  # 첫 번째 테이블이 S&P 500 구성 종목 목록


# 티커와 기업명 추출
ticker_Group= df['Symbol'].tolist()
company_names = df['Security'].tolist()

ticker_Group = [t.replace(".", "-") for t in ticker_Group]
company_names = [t.replace(".", "-") for t in company_names]
ticker_names = dict(zip(ticker_Group, company_names))

ticker_Group
ticker_names

In [ ]:
import pandas as pd
import yfinance as yf
import pandas_ta as ta
from datetime import datetime
import pytz

#오늘 날짜
today = datetime.now().date()
result_list = []

for ticker1 in ticker_Group:
    print(f"{ticker1}...")
    ticker = yf.Ticker(ticker1)
    df = ticker.history(period="2mo")
    # 기술적 지표 (단기형)
    df["RSI"] = ta.rsi(df["Close"], length=5).shift(1)
    bb = ta.bbands(df["Close"], length=10, std=2.0)
    for col in bb.columns:
        df[col] = bb[col].shift(1)
    df['SMA5'] = df['Close'].rolling(window=5).mean().shift(1)
    df['SMA10'] = df['Close'].rolling(window=10).mean().shift(1)
    df['BB_Dist'] = df['Close'] - df['BBL_10_2.0']

    macd = ta.macd(df["Close"], fast=6, slow=13, signal=4)
    for col in macd.columns:
        df[col] = macd[col].shift(1)

    stoch = ta.stoch(df["High"], df["Low"], df["Close"], k=5, d=3, smooth_k=1)
    for col in stoch.columns:
        df[col] = stoch[col].shift(1)

    df["OBV"] = ta.obv(df["Close"], df["Volume"]).shift(1)
    df["ATR"] = ta.atr(df["High"], df["Low"], df["Close"], length=14).shift(1)
    df["Golden_Cross"] = (
        (df["SMA5"].shift(1) < df["SMA10"].shift(1)) &
        (df["SMA5"] >= df["SMA10"])
    ).astype(int)
    df["ADX"] = ta.adx(df["High"], df["Low"], df["Close"]).iloc[:, 0].shift(1)
    df["CCI"] = ta.cci(df["High"], df["Low"], df["Close"], length=20).shift(1)
    df["ROC"] = ta.roc(df["Close"], length=10).shift(1)
    df["CMF"] = ta.cmf(df["High"], df["Low"], df["Close"], df["Volume"], length=20).shift(1)
    df["ADL"] = ta.ad(df["High"], df["Low"], df["Close"], df["Volume"]).shift(1)

    # 최신 행만
    last_row = df.iloc[-1]
    result_list.append({
        'Ticker': ticker1,
        'Date': today,
        'RSI': last_row['RSI'],
        'BB_Dist': last_row['BB_Dist'],
        'MACD_6_13_4': last_row['MACD_6_13_4'],
        'MACDh_6_13_4': last_row['MACDh_6_13_4'],
        'MACDs_6_13_4': last_row['MACDs_6_13_4'],
        'STOCHk_5_3_1': last_row['STOCHk_5_3_1'],
        'STOCHd_5_3_1': last_row['STOCHd_5_3_1'],
        'OBV': last_row['OBV'],
        'ATR': last_row['ATR'],
        'Golden_Cross': last_row['Golden_Cross'],
        'ADX': last_row['ADX'],
        'CCI': last_row['CCI'],
        'ROC': last_row['ROC'],
        'CMF': last_row['CMF'],
        'ADL': last_row['ADL']
    })

# 데이터프레임으로 변환
final_df = pd.DataFrame(result_list)

In [ ]:
import os
import pandas as pd
from datetime import date
from google.colab import drive
# 1. 구글 드라이브 마운트
drive.mount('/content/drive')

# 2. 오늘 날짜
today = date.today().strftime("%Y-%m-%d")
file_path = f"/content/drive/Othercomputers/내 Mac/Desktop/hateslop/프로젝트/예측데이터/({today})예측.csv"

df_result.to_csv(file_path, index=False)
print("✅ 저장 완료:", file_path)